<a href="https://colab.research.google.com/github/Ayushman125/Essentials_of_AI/blob/main/EAI_LAB_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Experiment 5




1) Write a python script to determine the size of a sample

>   a)Random

>  b)Cluster-based Sampling

>   c)Strade based Sampling

  write your own funcutions with required parameters.

# Sample Size Calculation Formulas

### 1. Simple Random Sampling
Determines sample size based on a desired confidence level and margin of error for a large or unknown population using Cochran's Formula:
$$n = \frac{Z^2 \cdot p \cdot (1 - p)}{e^2}$$

*Adjusted for a Finite Population ($N$):*
$$n_{adjusted} = \frac{n}{1 + \frac{n - 1}{N}}$$

### 2. Cluster-Based Sampling
Accounts for high intra-cluster correlation by inflating the simple random sample size using the **Design Effect (DEFF)**:
$$n_{cluster} = n \cdot [1 + (m - 1) \cdot \rho]$$
Where:
- $m$ = Average number of individuals per cluster.
- $\rho$ (rho) = Intra-cluster correlation coefficient (ICC).

### 3. Stratified Sampling
Allocates a total sample size across distinct strata using **Proportional Allocation**:
$$n_h = n \cdot \left( \frac{N_h}{N} \right)$$
Where:
- $N_h$ = Population size of stratum $h$.
- $N$ = Total population size ($N = \sum N_h$).
- $n$ = Total target sample size.

In [6]:
import math

try:
    from scipy.special import erfinv
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

# Standard Normal Z-scores for common confidence levels
Z_DICT = {
    80: 1.282,
    85: 1.440,
    90: 1.645,
    95: 1.960,
    99: 2.576
}

def calculate_z_score(confidence_level: float) -> float:
    """Calculates Z-score dynamically using Scipy if available, or falls back to lookup."""
    if confidence_level in Z_DICT:
        return Z_DICT[confidence_level]
    elif HAS_SCIPY:
        alpha = 1.0 - (confidence_level / 100.0)
        return round(math.sqrt(2) * erfinv(1 - alpha), 3)
    else:
        # Default fallback to 95% confidence level
        return 1.960


def random_sample_size(
    confidence_level: float = 95.0,
    margin_of_error: float = 0.05,
    proportion: float = 0.5,
    pop_size: int = None
) -> int:
    """Calculates sample size for Simple Random Sampling using Cochran's formula."""
    z = calculate_z_score(confidence_level)
    e = margin_of_error
    p = proportion

    # Base calculation for infinite population
    n0 = (z**2 * p * (1 - p)) / (e**2)

    # Finite population correction adjustment
    if pop_size and pop_size > 0:
        n = n0 / (1 + ((n0 - 1) / pop_size))
    else:
        n = n0

    return math.ceil(n)


def cluster_sample_size(
    confidence_level: float = 95.0,
    margin_of_error: float = 0.05,
    proportion: float = 0.5,
    icc: float = 0.05,
    cluster_size: int = None,
    num_clusters: int = None,
    pop_size: int = None
) -> dict:
    """Calculates cluster sample sizes based on Design Effect (DEFF)."""
    n_srs = random_sample_size(confidence_level, margin_of_error, proportion, pop_size)

    if cluster_size is not None and cluster_size > 0:
        deff = 1 + (cluster_size - 1) * icc
        total_raw = n_srs * deff
        k = math.ceil(total_raw / cluster_size)
        total_sample = k * cluster_size
        m = cluster_size
    elif num_clusters is not None and num_clusters > 0:
        k = num_clusters
        min_clusters_needed = math.ceil(n_srs * icc)

        # Check if requested clusters are too few to achieve desired precision
        if k <= (n_srs * icc):
            raise ValueError(
                f"Cannot achieve precision with only {k} clusters when ICC={icc}. "
                f"You need at least {min_clusters_needed + 1} clusters."
            )

        m = math.ceil((n_srs * (1 - icc)) / (k - (n_srs * icc)))
        deff = 1 + (m - 1) * icc
        total_sample = k * m
    else:
        raise ValueError("Either cluster_size or num_clusters must be specified.")

    return {
        "n_srs": n_srs,
        "deff": round(deff, 3),
        "num_clusters": k,
        "subjects_per_cluster": m,
        "total_sample_size": total_sample
    }


def stratified_sample_size(
    strata_populations: dict,
    total_target_sample: int = None,
    confidence_level: float = 95.0,
    margin_of_error: float = 0.05,
    proportion: float = 0.5
) -> dict:
    """Calculates proportional sample allocation across strata."""
    total_pop = sum(strata_populations.values())

    # If total target sample isn't provided, compute overall required SRS size first
    if total_target_sample is None:
        total_target_sample = random_sample_size(
            confidence_level, margin_of_error, proportion, pop_size=total_pop
        )

    strata_allocations = {}
    allocated_sum = 0

    for stratum, pop in strata_populations.items():
        weight = pop / total_pop
        n_h = math.floor(total_target_sample * weight)
        strata_allocations[stratum] = n_h
        allocated_sum += n_h

    # Distribute remaining sample units caused by floor rounding
    remainder = total_target_sample - allocated_sum
    for stratum in list(strata_allocations.keys())[:remainder]:
        strata_allocations[stratum] += 1

    return {
        "total_population": total_pop,
        "total_sample_size": total_target_sample,
        "allocations": strata_allocations
    }


# ==========================================
# CLI / Interactive Input Interface
# ==========================================

def main():
    print("=== SAMPLE SIZE CALCULATOR ===")
    print("1) Simple Random Sampling")
    print("2) Cluster-Based Sampling")
    print("3) Stratified Sampling")

    choice = input("\nEnter choice (1, 2, or 3): ").strip()

    if choice == '1':
        print("\n--- Simple Random Sampling ---")
        pop_str = input("Enter Total Population Size (press Enter if unknown/infinite): ").strip()
        pop_size = int(pop_str) if pop_str else None

        conf = float(input("Enter Confidence Level % [Default 95]: ") or 95)
        moe = float(input("Enter Margin of Error (e.g. 0.05) [Default 0.05]: ") or 0.05)
        p = float(input("Enter Estimated Proportion (e.g. 0.5) [Default 0.5]: ") or 0.5)

        n = random_sample_size(
            confidence_level=conf,
            margin_of_error=moe,
            proportion=p,
            pop_size=pop_size
        )
        print(f"\n>> Required Random Sample Size: {n} units")

    elif choice == '2':
        print("\n--- Cluster-Based Sampling ---")
        print("Specify design parameter constraint:")
        print("  a) Fixed Subjects per Cluster")
        print("  b) Fixed Total Number of Clusters")
        sub_choice = input("Choice (a/b): ").strip().lower()

        cluster_size, num_clusters = None, None
        if sub_choice == 'a':
            cluster_size = int(input("Enter target subjects per cluster (m): "))
        elif sub_choice == 'b':
            num_clusters = int(input("Enter available number of clusters (k): "))
        else:
            print("Invalid parameter selection.")
            return

        icc = float(input("Enter Intra-cluster Correlation Coefficient (ICC / rho, e.g., 0.02 - 0.05): "))
        pop_str = input("Enter Total Population Size (press Enter if unknown/infinite): ").strip()
        pop_size = int(pop_str) if pop_str else None

        conf = float(input("Enter Confidence Level % [Default 95]: ") or 95)
        moe = float(input("Enter Margin of Error [Default 0.05]: ") or 0.05)
        p = float(input("Enter Estimated Proportion [Default 0.5]: ") or 0.5)

        res = cluster_sample_size(
            confidence_level=conf,
            margin_of_error=moe,
            proportion=p,
            icc=icc,
            cluster_size=cluster_size,
            num_clusters=num_clusters,
            pop_size=pop_size
        )

        print(f"\n>> Baseline SRS Sample (n0): {res['n_srs']} units")
        print(f">> Design Effect (DEFF): {res['deff']}")
        print(f">> Required Number of Clusters (k): {res['num_clusters']}")
        print(f">> Subjects per Cluster (m): {res['subjects_per_cluster']}")
        print(f">> Total Cluster Sample Size: {res['total_sample_size']} units")

    elif choice == '3':
        print("\n--- Stratified Sampling ---")
        num_strata = int(input("Enter number of Strata: "))

        strata_pops = {}
        for i in range(num_strata):
            name = input(f"  Enter name for Stratum {i+1}: ") or f"Stratum_{i+1}"
            pop = int(input(f"  Enter population size for {name}: "))
            strata_pops[name] = pop

        target_str = input("Enter Total Target Sample Size (press Enter to auto-calculate from parameters): ").strip()

        if target_str:
            res = stratified_sample_size(
                strata_populations=strata_pops,
                total_target_sample=int(target_str)
            )
        else:
            conf = float(input("Enter Confidence Level % [Default 95]: ") or 95)
            moe = float(input("Enter Margin of Error [Default 0.05]: ") or 0.05)
            p = float(input("Enter Estimated Proportion [Default 0.5]: ") or 0.5)

            res = stratified_sample_size(
                strata_populations=strata_pops,
                confidence_level=conf,
                margin_of_error=moe,
                proportion=p
            )

        print(f"\n>> Total Population Across Strata: {res['total_population']}")
        print(f">> Total Allocated Sample Size: {res['total_sample_size']}")
        print(">> Proportional Sample Allocation per Stratum:")
        for stratum, count in res["allocations"].items():
            print(f"   - {stratum}: {count} units")

    else:
        print("Invalid selection.")

if __name__ == "__main__":
    main()

=== SAMPLE SIZE CALCULATOR ===
1) Simple Random Sampling
2) Cluster-Based Sampling
3) Stratified Sampling

Enter choice (1, 2, or 3): 2

--- Cluster-Based Sampling ---
Specify design parameter constraint:
  a) Fixed Subjects per Cluster
  b) Fixed Total Number of Clusters
Choice (a/b): b
Enter available number of clusters (k): 5
Enter Intra-cluster Correlation Coefficient (ICC / rho, e.g., 0.02 - 0.05): 0.05
Enter Total Population Size (press Enter if unknown/infinite):  
Enter Confidence Level % [Default 95]: 95
Enter Margin of Error [Default 0.05]: 0.05
Enter Estimated Proportion [Default 0.5]: 0.05

>> Baseline SRS Sample (n0): 73 units
>> Design Effect (DEFF): 3.55
>> Required Number of Clusters (k): 5
>> Subjects per Cluster (m): 52
>> Total Cluster Sample Size: 260 units


2. For a given population of size 'n' (generate using random numbers), select 'k'(k<<<n) sample of size n' (n'<n) and find the best sample among 'k' sample by measuring goodness (MOCT,MOS) of samples.

# Sample Goodness Evaluation Formulas

To determine the "best" sample out of $k$ drawn samples, we calculate the absolute relative error between each sample's statistical properties and the underlying population properties.

### 1. Measures of Central Tendency (MOCT) Error
Evaluates how well the sample mean ($\bar{x}$) represents the population mean ($\mu$):
$$\text{Error}_{\text{MOCT}} = \left| \frac{\bar{x} - \mu}{\mu} \right|$$

### 2. Measures of Spread (MOS) Error
Evaluates how well the sample standard deviation ($s$) represents the population standard deviation ($\sigma$):
$$\text{Error}_{\text{MOS}} = \left| \frac{s - \sigma}{\sigma} \right|$$

### 3. Total Goodness Score (Loss Metric)
Combines MOCT and MOS into a single goodness error metric (where lower values indicate a better, more representative sample):
$$\text{Score}_{\text{Goodness}} = w_1 \cdot \text{Error}_{\text{MOCT}} + w_2 \cdot \text{Error}_{\text{MOS}}$$
*(By default, equal weights $w_1 = 0.5$ and $w_2 = 0.5$ are applied).*

In [4]:
import numpy as np
import pandas as pd

def generate_population(N, mean=100, std_dev=15, seed=42):
    """Generates a synthetic population of size N following a normal distribution."""
    np.random.seed(seed)
    population = np.random.normal(loc=mean, scale=std_dev, size=N)
    return population


def calculate_moct_mos(data):
    """Calculates Measures of Central Tendency (Mean) and Measures of Spread (Std Dev)."""
    moct = np.mean(data)
    mos = np.std(data, ddof=1) if len(data) > 1 else 0.0  # Sample standard deviation
    return moct, mos


def evaluate_samples(population, k, sample_size, w_moct=0.5, w_mos=0.5):
    """
    Selects k samples of size sample_size from population and finds the best sample
    based on MOCT and MOS goodness metrics.
    """
    # Population benchmark statistics
    pop_moct, pop_mos = calculate_moct_mos(population)

    results = []
    samples_data = []

    for i in range(k):
        # Draw random sample without replacement
        sample = np.random.choice(population, size=sample_size, replace=False)
        samples_data.append(sample)

        sample_moct, sample_mos = calculate_moct_mos(sample)

        # Calculate Relative Errors
        moct_error = abs(sample_moct - pop_moct) / abs(pop_moct)
        mos_error = abs(sample_mos - pop_mos) / abs(pop_mos)

        # Combined Goodness Error (Lower is better)
        total_goodness_error = (w_moct * moct_error) + (w_mos * mos_error)

        results.append({
            'Sample_ID': i + 1,
            'Sample_Mean (MOCT)': round(sample_moct, 4),
            'Sample_StdDev (MOS)': round(sample_mos, 4),
            'MOCT_Error': round(moct_error, 6),
            'MOS_Error': round(mos_error, 6),
            'Goodness_Score': round(total_goodness_error, 6)
        })

    results_df = pd.DataFrame(results)

    # Best sample has the minimum combined goodness error
    best_idx = results_df['Goodness_Score'].idxmin()
    best_sample_info = results_df.loc[best_idx]
    best_sample_data = samples_data[best_idx]

    benchmark_info = {
        'Pop_Mean (MOCT)': round(pop_moct, 4),
        'Pop_StdDev (MOS)': round(pop_mos, 4)
    }

    return results_df, best_sample_info, best_sample_data, benchmark_info


def main():
    print("=== SAMPLE GOODNESS EVALUATOR (MOCT & MOS) ===")

    # User Inputs
    N = int(input("Enter Population Size N (e.g., 10000): ") or 10000)
    k = int(input("Enter number of samples k to select (e.g., 10): ") or 10)
    n_prime = int(input(f"Enter sample size n' (n' < {N}, e.g., 100): ") or 100)

    if n_prime >= N:
        print("Error: Sample size n' must be strictly less than Population size N.")
        return

    # Generate synthetic population
    print("\nGenerating population...")
    population = generate_population(N)

    # Evaluate samples
    results_df, best_sample_info, best_sample_data, benchmark_info = evaluate_samples(
        population=population,
        k=k,
        sample_size=n_prime
    )

    # Display Results
    print("\n--- POPULATION BENCHMARKS ---")
    print(f"Population Mean (MOCT)   : {benchmark_info['Pop_Mean (MOCT)']}")
    print(f"Population StdDev (MOS) : {benchmark_info['Pop_StdDev (MOS)']}")

    print("\n--- SAMPLES EVALUATION TABLE ---")
    print(results_df.to_string(index=False))

    print("\n" + "="*45)
    print(f"   BEST SAMPLE: Sample #{int(best_sample_info['Sample_ID'])}")
    print("="*45)
    print(f"Mean (MOCT)       : {best_sample_info['Sample_Mean (MOCT)']}")
    print(f"StdDev (MOS)      : {best_sample_info['Sample_StdDev (MOS)']}")
    print(f"MOCT Error        : {best_sample_info['MOCT_Error'] * 100:.4f}%")
    print(f"MOS Error         : {best_sample_info['MOS_Error'] * 100:.4f}%")
    print(f"Combined Loss     : {best_sample_info['Goodness_Score']}")

# Execute interactive session
if __name__ == "__main__":
    main()

=== SAMPLE GOODNESS EVALUATOR (MOCT & MOS) ===
Enter Population Size N (e.g., 10000): 1000
Enter number of samples k to select (e.g., 10): 10
Enter sample size n' (n' < 1000, e.g., 100): 500

Generating population...

--- POPULATION BENCHMARKS ---
Population Mean (MOCT)   : 100.29
Population StdDev (MOS) : 14.6882

--- SAMPLES EVALUATION TABLE ---
 Sample_ID  Sample_Mean (MOCT)  Sample_StdDev (MOS)  MOCT_Error  MOS_Error  Goodness_Score
         1            100.5419              14.8314    0.002512   0.009746        0.006129
         2            100.4546              14.6491    0.001642   0.002663        0.002152
         3            100.2891              15.0052    0.000009   0.021578        0.010793
         4            100.8111              14.2483    0.005197   0.029952        0.017574
         5            100.6285              14.9931    0.003376   0.020756        0.012066
         6            100.2288              15.2195    0.000610   0.036168        0.018389
         7   